<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/06-Function-Calling-%26-Data_Extraction/04_Structured_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q groq

from groq import Groq
from google.colab import userdata
import inspect, json
import requests

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.2 MB/s eta 0:00:00


In [3]:
def get_completion(prompt, model="llama-3.3-70b-versatile"):
  messages = [{'role':'system','content': "You are being used as a tool calling agent, just respond with the function call ONLY"},
              {"role": "user", "content": prompt}]
  response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,)
  return response.choices[0].message.content

In [4]:
def get_completion_from_messages(messages, tools,
                                 model="llama-3.3-70b-versatile",
                                 temperature=0, ):
    response = client.chat.completions.create(
        model=model,
        tools=tools,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.tool_calls[0]

In [5]:
def build_prompt(function_list, user_query):
  f_prompt = ""
  for function in function_list:
    signature = inspect.signature(function)
    docstring = function.__doc__
    prompt= f''' Function:
def {function.__name__}{signature}
  """
  {docstring.strip()}
  """
     '''
    f_prompt += prompt
  f_prompt+= f"User Query: {user_query}<human_end>"
  return f_prompt

## Address Extraction

In [1]:
data_paragraph = (
    "Let's consider a small community where several individuals reside. "
    "First, we have Sarah Johnson, whose home is located at 123 Maple Drive, Springfield, IL 62704. She often walks her golden retriever, Buster, in the mornings. "
    "Just next door, her neighbor, David Lee, a retired professor, can be found at 456 Oak Avenue, also in Springfield, IL 62704. He enjoys gardening. "
    "A little further down the street, Emily Chen has her residence at 789 Pine Lane, Springfield, IL 62704. She recently started a local book club. "
    "Across town, Robert Miller, a software engineer, resides at 55 Elmwood Drive, Springfield, IL 62704, and is known for his elaborate Halloween decorations. "
    "Moving to a different locality, John Smith lives at 101 Elm Street, Rivertown, NY 10001, where he runs a small antique shop. His shop's phone number is (555) 123-4567. "
    "His business partner, Jessica White, has an apartment at 33 Willow Street, Rivertown, NY 10001. "
    "Lastly, Maria Rodriguez's address is 202 Birch Road, Hilltop, CA 90210. She recently adopted a cat named Whiskers. "
    "Her cousin, Carlos Garcia, lives at 99 Ocean View, Seaside, CA 90222, and is a passionate surfer. "
)
print(data_paragraph)

Let's consider a small community where several individuals reside. First, we have Sarah Johnson, whose home is located at 123 Maple Drive, Springfield, IL 62704. She often walks her golden retriever, Buster, in the mornings. Just next door, her neighbor, David Lee, a retired professor, can be found at 456 Oak Avenue, also in Springfield, IL 62704. He enjoys gardening. A little further down the street, Emily Chen has her residence at 789 Pine Lane, Springfield, IL 62704. She recently started a local book club. Across town, Robert Miller, a software engineer, resides at 55 Elmwood Drive, Springfield, IL 62704, and is known for his elaborate Halloween decorations. Moving to a different locality, John Smith lives at 101 Elm Street, Rivertown, NY 10001, where he runs a small antique shop. His shop's phone number is (555) 123-4567. His business partner, Jessica White, has an apartment at 33 Willow Street, Rivertown, NY 10001. Lastly, Maria Rodriguez's address is 202 Birch Road, Hilltop, CA 9

In [6]:
def address_name_pairs(names: list[str], addresses: list[str]):
  """Give names and associated addresses"""
  for name, addr in zip(names,addresses):
    print(name, ": ",addr)


In [7]:
query= build_prompt([address_name_pairs],data_paragraph)
response = get_completion(query)
exec(response)

Sarah Johnson :  123 Maple Drive, Springfield, IL 62704
David Lee :  456 Oak Avenue, Springfield, IL 62704
Emily Chen :  789 Pine Lane, Springfield, IL 62704
Robert Miller :  55 Elmwood Drive, Springfield, IL 62704
John Smith :  101 Elm Street, Rivertown, NY 10001
Jessica White :  33 Willow Street, Rivertown, NY 10001
Maria Rodriguez :  202 Birch Road, Hilltop, CA 90210
Carlos Garcia :  99 Ocean View, Seaside, CA 90222


In [10]:
from dataclasses import dataclass
@dataclass
class Record:
  name: str
  addresses: list[str]


In [9]:
prompt=f'''
@dataclass
class Record:
  name: str
  addresses: list[str]
Function:
def insert_into_database(names: list[Record]):
  """Inserts the records into the database"""
{data_paragraph}<human_end>
'''
response = get_completion(prompt)
print(response)

insert_into_database([Record("Sarah Johnson", ["123 Maple Drive, Springfield, IL 62704"]), Record("David Lee", ["456 Oak Avenue, Springfield, IL 62704"]), Record("Emily Chen", ["789 Pine Lane, Springfield, IL 62704"]), Record("Robert Miller", ["55 Elmwood Drive, Springfield, IL 62704"]), Record("John Smith", ["101 Elm Street, Rivertown, NY 10001"]), Record("Jessica White", ["33 Willow Street, Rivertown, NY 10001"]), Record("Maria Rodriguez", ["202 Birch Road, Hilltop, CA 90210"]), Record("Carlos Garcia", ["99 Ocean View, Seaside, CA 90222"])])


## Generating Valid JSONs

{
  "city_name" : "London"
  "location" : {
      "country" : "United Kingdom",
      "continent" : {
          "simple_name" : "Europe",
          "other_name" : "Afro-Eur-Asia"
      }
  }
}

In [12]:
def city_info(city_name : str, location : dict):
  """
  Gets the city info
  """
  return locals()
def construct_location_dict(country : str, continent : dict):
  """
  Provides the location dictionary
  """
  return locals()
def construct_continent_dict(simple_name : str, other_name : str):
  """
  Provides the continent dict
  """
  return locals()

In [13]:
prompt = \
'''
Function:
def city_info(city_name : str, location : dict):
"""
Gets the city info
"""

Function:
def construct_location_dict(country : str, continent : dict):
"""
Provides the location dictionary
"""

def construct_continent_dict(simple_name : str, other_name : str):
"""
Provides the continent dict
"""

User Query: {question}<human_end>
'''

In [14]:
question = "I want the city info for London, "\
"which is in the United Kingdom, which is in Europe or Afro-Eur-Asia."

output = get_completion(prompt.format(question = question))
json0 = eval(output)
print (json0)

{'city_name': 'London', 'location': {'country': 'United Kingdom', 'continent': {'simple_name': 'Europe', 'other_name': 'Afro-Eur-Asia'}}}


In [15]:
json.dumps(json0)

'{"city_name": "London", "location": {"country": "United Kingdom", "continent": {"simple_name": "Europe", "other_name": "Afro-Eur-Asia"}}}'